# 03 — Constraints, Examples, and Few-Shot Learning

Northstar needs to route support requests safely. This credential-free lab treats examples as a context-selection decision, not magic prompt decoration.

## Objectives and safety

Compare no, static, random, similarity, and diversity example strategies on the same frozen cases. The simulator is transparent and offline; it is not a model benchmark or a provider claim.

## Mental model

`observed failure → small boundary example hypothesis → select examples → evaluate held-out cases → retain or remove`

Examples help only when their measured benefit exceeds their token, latency, and regression cost.

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

path = Path.cwd() / 'curriculum/beginner/03-constraints-examples-and-few-shot-learning/lab.py'
if not path.exists(): path = Path.cwd() / 'lab.py'
spec = spec_from_file_location('few_shot_lab', path)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print({'examples': len(lab.EXAMPLES), 'cases': len(lab.CASES)})

{'examples': 6, 'cases': 4}


## Baseline: no examples

The direct lexical baseline makes the initial decision rule visible. It may already be sufficient for clear cases; adding examples must prove incremental value.

In [2]:
baseline = lab.experiment('none')
lab.metrics(baseline), baseline

({'accuracy': 1.0, 'mean_example_tokens': 0.0},
 [{'case': 'My package has not arrived.',
   'expected': 'shipping',
   'predicted': 'shipping',
   'correct': True,
   'examples': (),
   'token_estimate': 0},
  {'case': 'I want to send back the product.',
   'expected': 'refund',
   'predicted': 'refund',
   'correct': True,
   'examples': (),
   'token_estimate': 0},
  {'case': 'Update my account email address.',
   'expected': 'account',
   'predicted': 'account',
   'correct': True,
   'examples': (),
   'token_estimate': 0},
  {'case': 'There is a payment problem.',
   'expected': 'unknown',
   'predicted': 'unknown',
   'correct': True,
   'examples': (),
   'token_estimate': 0}])

## Experiment: compare selection strategies

Freeze the case set and change only example selection. Static examples are predictable, random selection tests accidental relevance, similarity selection prefers local matches, and diversity selection prevents duplicate labels.

In [3]:
strategies = ['none', 'static', 'random', 'similarity', 'diversity']
results = {strategy: lab.experiment(strategy) for strategy in strategies}
comparison = {strategy: lab.metrics(rows) for strategy, rows in results.items()}
comparison

{'none': {'accuracy': 1.0, 'mean_example_tokens': 0.0},
 'static': {'accuracy': 0.75, 'mean_example_tokens': 12.0},
 'random': {'accuracy': 1.0, 'mean_example_tokens': 11.0},
 'similarity': {'accuracy': 0.75, 'mean_example_tokens': 11.25},
 'diversity': {'accuracy': 0.75, 'mean_example_tokens': 11.75}}

## Inspect selected context

Selection is itself observable. Check which examples were supplied to the ambiguous payment case before attributing a behavior change to the model.

In [4]:
{strategy: rows[-1] for strategy, rows in results.items()}

{'none': {'case': 'There is a payment problem.',
  'expected': 'unknown',
  'predicted': 'unknown',
  'correct': True,
  'examples': (),
  'token_estimate': 0},
 'static': {'case': 'There is a payment problem.',
  'expected': 'unknown',
  'predicted': 'shipping',
  'correct': False,
  'examples': (('Where is my delivery?', 'shipping'),
   ('Can I return an item that arrived yesterday?', 'refund')),
  'token_estimate': 12},
 'random': {'case': 'There is a payment problem.',
  'expected': 'unknown',
  'predicted': 'unknown',
  'correct': True,
  'examples': (('Can I return an item that arrived yesterday?', 'refund'),
   ('Track package 99.', 'shipping')),
  'token_estimate': 11},
 'similarity': {'case': 'There is a payment problem.',
  'expected': 'unknown',
  'predicted': 'shipping',
  'correct': False,
  'examples': (('Where is my delivery?', 'shipping'),
   ('The product is damaged; how do I return it?', 'refund')),
  'token_estimate': 13},
 'diversity': {'case': 'There is a payment p

## Failure injection: harmful bias

A refund-like example can be similar to a payment complaint while still being the wrong precedent. The safe `unknown` label is a required boundary, not an error to optimize away.

In [5]:
payment = lab.Case('There is a payment problem.', 'unknown')
selected = lab.select_examples('similarity', payment.text)
predicted = lab.classify(payment.text, selected)
assert predicted != 'unknown'  # controlled harmful-bias failure
{'selected': selected, 'failure_prediction': predicted, 'expected': payment.expected}

{'selected': (Example(text='Where is my delivery?', label='shipping'),
  Example(text='The product is damaged; how do I return it?', label='refund')),
 'failure_prediction': 'shipping',
 'expected': 'unknown'}

## Evaluation, production upgrade, and exercises

Use development cases to refine examples and held-out cases to decide whether to ship. Record example IDs, selector version, token use, latency, and accuracy by slice. Apply access controls before retrieval; an example is context and can leak or poison behavior.

1. Add a contradictory example and observe the regression.
2. Implement a tenant metadata filter.
3. Define when an empty example set is the best choice.
4. Challenge: design a release gate that balances accuracy, `unknown` correctness, and context cost.

**Summary:** example selection is an evaluation problem, not an aesthetic prompt choice.